# Explainability and Error Analysis (FashionStyle14)

## Objective
This notebook runs two analysis tracks on trained fashion style classifiers:

- **Error analysis:** confusion matrices for all baseline models (excluding the proposed cross-attention model), class-pair confusion for cross-attention and image-only models, and Grad-CAM on CLIP vision encoders.

## Scope
| Track | Models |
|-------|--------|
| Part 1 - all baselines | image-only, multimodal, text-only (**excludes proposed `cross_attention`**) |
| Part 2 - class-pair confusion | `cross_attention` + image-only (`clip`, `convnext`, `swin`, `vit`) |
| Part 3 - Grad-CAM | `clip` (image-only) + `cross_attention` (CLIP ViT-B/32 vision encoder) |

## Inputs (explicit paths)
| Resource | Path |
|----------|------|
| Image list | `FashionStyle14_v1/complete_dataset.csv` |
| Captions | `FashionStyle14_v1/caption/fashion_captions_llava_success.csv` |
| Split seeds | `FashionStyle14_v1/seeds_list.txt` (first 10 seeds) |
| Images | `FashionStyle14_v1/dataset/` |
| Cross-attention checkpoints | `results/proposed/cross_attention/seed_<seed>/best_model.pt` |
| Image-only checkpoints | `results/image_only/<model>/seed_<seed>/best_model.pt` |
| Multimodal checkpoints | `results/multimodal/<model>/seed_<seed>/best_model.pt` |
| Text-only checkpoints | `results/text_only/fashionbert/seed_<seed>/best_model.pt` |

Checkpoint files are referenced directly. If a file is missing at runtime, loading is skipped with a warning and weights remain randomly initialized (for pipeline smoke tests).

## Outputs
All artifacts are written under **`results/explainability/`** with one subfolder per technique:

| Subfolder | Contents |
|-----------|----------|
| `attention_weights/` | Sample-level image/text token attention bar plots |
| `per_class_attention/` | Mean cross-attention maps per style class |
| `modality_contribution/` | Confidence drop when removing image or text |
| `confusion_matrix/` | Per-model heatmaps, top confused pairs, global summary |
| `class_pair_confusion/` | Symmetric confusion and top pairs (cross-attention + image-only) |
| `gradcam/` | Success/failure Grad-CAM overlays on fashion images |

## Notes
- Grad-CAM (Part 3) run on **all 10 robustness seeds**
- Confusion analysis (Part 1 and 2) also aggregates all seeds.

## 1. Configuration, imports, and paths


In [ ]:
from __future__ import annotations

import json
import os
import random
import re
import warnings
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    BlipModel,
    BlipProcessor,
    CLIPModel,
    CLIPProcessor,
    ViltModel,
    ViltProcessor,
)

warnings.filterwarnings("ignore", category=UserWarning)
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Hyperparameters aligned with training notebooks
BATCH_SIZE = 8
MAX_SEQ_LENGTH = 128
MAX_SEQ_LENGTH_VILT = 40
DROPOUT = 0.5
MODEL_INIT_SEED = 42
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.7, 0.15, 0.15
NUM_SEEDS_TO_USE = 10
NUM_VIZ_SAMPLES_PER_SEED = 8  # sample-level attention plots per seed (set to None for entire test set)
TOP_K_CONFUSED_PAIRS = 15
GRADCAM_NUM_CASES = 6

CLIP_MODEL_ID = "openai/clip-vit-base-patch32"
BERT_MODEL_ID = "bert-base-uncased"
BLIP_MODEL_ID = "Salesforce/blip-image-captioning-base"
VILT_MODEL_ID = "dandelin/vilt-b32-mlm"
VIT_MODEL_NAME = "vit_base_patch16_224"
CONVNEXT_MODEL_NAME = "convnext_base"
SWIN_MODEL_NAME = "swin_base_patch4_window7_224"

CROSS_ATTN_DIM = 512
CROSS_ATTN_HEADS = 8
CROSS_ATTN_DROPOUT = 0.1

IMAGE_ONLY_MODELS = ["clip", "convnext", "swin", "vit"]
MULTIMODAL_MODELS = ["concat_mlp", "gated_fusion", "blip", "vilbert"]
TEXT_ONLY_MODELS = ["fashionbert"]
PROPOSED_MODEL_KEY = "cross_attention"  # excluded from Part B.1 "all models"

ALL_BASELINE_MODELS: Dict[str, Tuple[str, str]] = {
    **{m: ("image_only", m) for m in IMAGE_ONLY_MODELS},
    **{m: ("multimodal", m) for m in MULTIMODAL_MODELS},
    **{m: ("text_only", m) for m in TEXT_ONLY_MODELS},
}
CLASS_PAIR_MODELS = [PROPOSED_MODEL_KEY] + IMAGE_ONLY_MODELS
# Grad-CAM uses the CLIP ViT vision encoder (clip image-only + cross-attention)
GRADCAM_MODELS = ["clip", PROPOSED_MODEL_KEY]

OUTPUT_ROOT = Path("results/explainability")
DIR_ATTENTION = OUTPUT_ROOT / "attention_weights"
DIR_PER_CLASS_ATTN = OUTPUT_ROOT / "per_class_attention"
DIR_MODALITY = OUTPUT_ROOT / "modality_contribution"
DIR_CONFUSION = OUTPUT_ROOT / "confusion_matrix"
DIR_CLASS_PAIR = OUTPUT_ROOT / "class_pair_confusion"
DIR_GRADCAM = OUTPUT_ROOT / "gradcam"
for d in [DIR_ATTENTION, DIR_PER_CLASS_ATTN, DIR_MODALITY, DIR_CONFUSION, DIR_CLASS_PAIR, DIR_GRADCAM]:
    d.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Output root:", OUTPUT_ROOT.resolve())
print("Baseline models (Part B.1):", list(ALL_BASELINE_MODELS.keys()))
print("Excluded proposed model:", PROPOSED_MODEL_KEY)

## 2. Dataset loading and split helpers


In [ ]:
def resolve_paths() -> Tuple[Path, Path, Path]:
    cwd = Path.cwd().resolve()
    for root in [cwd, cwd / "FusionStyle", cwd.parent / "FusionStyle"]:
        data_dir = root / "FashionStyle14_v1"
        if (data_dir / "complete_dataset.csv").is_file():
            return root, data_dir, data_dir / "caption" / "fashion_captions_llava_success.csv"
    raise FileNotFoundError("FashionStyle14_v1 not found; run from FusionStyle/.")


def load_seeds(seeds_file: Path) -> List[int]:
    content = seeds_file.read_text(encoding="utf-8")
    matches = re.findall(r"Seed\s+(\d+)", content, flags=re.IGNORECASE)
    return sorted({int(s) for s in matches})[:NUM_SEEDS_TO_USE]


def normalize_rel_path(path_str: str) -> str:
    return str(path_str).strip().replace("\\", "/")


def canonical_merge_key(raw: str, image_root: Path) -> str:
    s = normalize_rel_path(raw).lstrip("./")
    low = s.lower()
    if low.startswith("fashionstyle14_v1/"):
        s = s[len("fashionstyle14_v1/") :].lstrip("/")
        low = s.lower()
    marker = "dataset/"
    ix = low.find(marker)
    if ix >= 0:
        return normalize_rel_path(s[ix:])
    p = Path(s)
    if p.is_absolute():
        try:
            rel = Path(p.resolve()).relative_to(image_root.resolve())
            return normalize_rel_path(str(rel).replace(os.sep, "/"))
        except ValueError:
            pass
    return s


def load_image_only_frame(csv_path: Path, image_root: Path) -> pd.DataFrame:
    lines = csv_path.read_text(encoding="utf-8").splitlines()
    rel = [ln.strip() for ln in lines if ln.strip()]
    df = pd.DataFrame({"rel_path": rel})
    df["rel_path"] = df["rel_path"].map(normalize_rel_path)
    df["merge_key"] = df["rel_path"].map(lambda r: canonical_merge_key(r, image_root))
    df["style"] = df["merge_key"].str.split("/").str[1]
    df["abs_path"] = df["rel_path"].apply(lambda r: str((image_root / r.replace("/", os.sep)).resolve()))
    return df[df["abs_path"].map(os.path.isfile)].reset_index(drop=True)


def load_captions(path: Path, image_root: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8")
    if "status" in df.columns:
        df = df[df["status"].astype(str).str.lower() == "success"]
    path_col = next(c for c in df.columns if c.lower().strip() in {"image_path", "rel_path", "path", "filename"})
    cap_col = next(c for c in df.columns if "caption" in c.lower() or c.lower() in {"text", "description"})
    out = df[[path_col, cap_col]].rename(columns={path_col: "raw_image_path", cap_col: "caption"})
    out["merge_key"] = out["raw_image_path"].map(lambda r: canonical_merge_key(r, image_root))
    out["caption"] = out["caption"].fillna("").astype(str).str.strip()
    return out[out["caption"] != ""].drop_duplicates("merge_key", keep="last")


PROJECT_ROOT, DATA_DIR, CAPTION_CSV = resolve_paths()
IMAGE_ROOT = DATA_DIR
COMPLETE_CSV = DATA_DIR / "complete_dataset.csv"
SEEDS = load_seeds(DATA_DIR / "seeds_list.txt")

df_images = load_image_only_frame(COMPLETE_CSV, IMAGE_ROOT)
cap_df = load_captions(CAPTION_CSV, IMAGE_ROOT)
df_mm = df_images.merge(cap_df[["merge_key", "caption"]], on="merge_key", how="inner").reset_index(drop=True)

classes = sorted(df_images["style"].unique().tolist())
assert len(classes) == 14
style_to_idx = {s: i for i, s in enumerate(classes)}
idx_to_style = {i: s for s, i in style_to_idx.items()}
num_classes = len(classes)


def split_by_seed(df: pd.DataFrame, seed_value: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df, temp_df = train_test_split(df, test_size=VAL_RATIO + TEST_RATIO, stratify=df["style"], random_state=seed_value)
    val_df, test_df = train_test_split(temp_df, test_size=TEST_RATIO / (VAL_RATIO + TEST_RATIO), stratify=temp_df["style"], random_state=seed_value)
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


print("PROJECT_ROOT:", PROJECT_ROOT)
print("Samples (image-only):", len(df_images))
print("Samples (with captions):", len(df_mm))
print("Classes:", classes)
print("Seeds:", SEEDS)

## 3. Shared utilities (checkpoints, confusion, plotting)


In [ ]:
def checkpoint_path(category: str, model_key: str, seed: int) -> Path:
    if category == "proposed":
        return PROJECT_ROOT / "results" / "proposed" / model_key / f"seed_{seed}" / "best_model.pt"
    return PROJECT_ROOT / "results" / category / model_key / f"seed_{seed}" / "best_model.pt"


def load_weights(model: nn.Module, ckpt_path: Path) -> None:
    print(f"Loading checkpoint: {ckpt_path}")
    if not ckpt_path.is_file():
        print(f"  WARNING: checkpoint not found; using current weights (pretend path for pipeline).")
        return
    state = torch.load(ckpt_path, map_location=device, weights_only=True)
    model.load_state_dict(state, strict=False)
    print("  Loaded successfully.")


def top_confused_pairs(cm: np.ndarray, k: int = TOP_K_CONFUSED_PAIRS) -> pd.DataFrame:
    n = cm.shape[0]
    rows = []
    for i in range(n):
        for j in range(n):
            if i != j and cm[i, j] > 0:
                rows.append({
                    "true_class": idx_to_style[i],
                    "pred_class": idx_to_style[j],
                    "count": int(cm[i, j]),
                    "rate_given_true": float(cm[i, j] / max(cm[i].sum(), 1)),
                })
    df = pd.DataFrame(rows).sort_values("count", ascending=False).head(k).reset_index(drop=True)
    return df


def symmetric_confusion_rates(cm: np.ndarray) -> pd.DataFrame:
    pairs = []
    n = cm.shape[0]
    for i in range(n):
        for j in range(i + 1, n):
            a, b = cm[i, j], cm[j, i]
            if a + b == 0:
                continue
            pairs.append({
                "class_a": idx_to_style[i],
                "class_b": idx_to_style[j],
                "a_to_b": int(a),
                "b_to_a": int(b),
                "symmetric_total": int(a + b),
                "symmetric_rate": float((a + b) / max(cm.sum(), 1)),
            })
    return pd.DataFrame(pairs).sort_values("symmetric_total", ascending=False).reset_index(drop=True)


def plot_confusion_heatmap(cm: np.ndarray, title: str, out_path: Path) -> None:
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cm, annot=False, cmap="Blues", xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show()
    plt.close(fig)
    print(f"Saved heatmap: {out_path}")



def build_classifier_head(in_dim: int, num_classes: int) -> nn.Sequential:
    """Shared MLP classifier head (512/768/etc -> 256 -> 128 -> num_classes)."""
    return nn.Sequential(
        nn.Linear(in_dim, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(DROPOUT),
        nn.Linear(256, 128),
        nn.ReLU(inplace=True),
        nn.Dropout(DROPOUT),
        nn.Linear(128, num_classes),
    )


_BACKBONE_CACHE: Dict[str, Any] = {}


def get_clip_bert():
    if "clip" not in _BACKBONE_CACHE:
        _BACKBONE_CACHE["clip"] = CLIPModel.from_pretrained(CLIP_MODEL_ID)
        _BACKBONE_CACHE["clip_p"] = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
        _BACKBONE_CACHE["bert"] = AutoModel.from_pretrained(BERT_MODEL_ID)
        _BACKBONE_CACHE["bert_t"] = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    return (
        _BACKBONE_CACHE["clip"],
        _BACKBONE_CACHE["clip_p"],
        _BACKBONE_CACHE["bert"],
        _BACKBONE_CACHE["bert_t"],
    )


def _batch_labels(batch: Dict[str, Any]) -> torch.Tensor:
    """Accept batches from image-only or multimodal loaders."""
    if "labels" in batch:
        lab = batch["labels"]
    elif "label" in batch:
        lab = batch["label"]
    else:
        raise KeyError("batch must contain 'labels' or 'label'")
    if not isinstance(lab, torch.Tensor):
        lab = torch.as_tensor(lab, dtype=torch.long)
    if lab.dim() == 0:
        lab = lab.unsqueeze(0)
    return lab


def _model_batch(batch: Dict[str, Any]) -> Dict[str, Any]:
    """Forward-pass tensors only (skip metadata keys)."""
    skip = {"label", "labels", "abs_path", "abs_paths"}
    out: Dict[str, Any] = {}
    for k, v in batch.items():
        if k in skip:
            continue
        out[k] = v.to(device) if torch.is_tensor(v) else v
    return out


def predict_loader(model, loader, forward_fn) -> Tuple[np.ndarray, np.ndarray]:
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            labels = _batch_labels(batch).to(device)
            logits = forward_fn(model, _model_batch(batch))
            pred = logits.argmax(dim=1)
            ys.extend(labels.cpu().tolist())
            ps.extend(pred.cpu().tolist())
    return np.array(ys), np.array(ps)

print("Shared utilities ready.")


## Error Analysis

### Cross-attention model with attention weight export
The fusion module returns averaged multi-head attention weights from image query to text keys.

In [ ]:
class FashionMultiModalFrameDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, style_to_idx: Dict[str, int]):
        self.frame = frame.reset_index(drop=True)
        self.style_to_idx = style_to_idx

    def __len__(self) -> int:
        return len(self.frame)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.frame.iloc[idx]
        return {"abs_path": row["abs_path"], "caption": str(row["caption"]), "labels": torch.tensor(self.style_to_idx[row["style"]], dtype=torch.long)}


class CrossAttentionFusion(nn.Module):
    def __init__(self, visual_dim: int, textual_dim: int, d_model: int = CROSS_ATTN_DIM, nhead: int = CROSS_ATTN_HEADS, dropout: float = CROSS_ATTN_DROPOUT):
        super().__init__()
        self.visual_proj = nn.Linear(visual_dim, d_model)
        self.text_proj = nn.Linear(textual_dim, d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, visual_feat, text_tokens, key_padding_mask=None, return_attn: bool = False):
        query = self.visual_proj(visual_feat).unsqueeze(1)
        key_value = self.text_proj(text_tokens)
        attn_out, attn_w = self.cross_attn(query, key_value, key_value, key_padding_mask=key_padding_mask, need_weights=True, average_attn_weights=True)
        fused = self.norm(attn_out.squeeze(1))
        if return_attn:
            return fused, attn_w.squeeze(1)
        return fused


def build_classifier_head(in_dim: int, num_classes: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(in_dim, 256), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
        nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
        nn.Linear(128, num_classes),
    )


_CACHE: Dict[str, Any] = {}


def get_clip_bert():
    if "clip" not in _CACHE:
        _CACHE["clip"] = CLIPModel.from_pretrained(CLIP_MODEL_ID)
        _CACHE["clip_p"] = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
        _CACHE["bert"] = AutoModel.from_pretrained(BERT_MODEL_ID)
        _CACHE["bert_t"] = AutoTokenizer.from_pretrained(BERT_MODEL_ID)
    return _CACHE["clip"], _CACHE["clip_p"], _CACHE["bert"], _CACHE["bert_t"]


class ClipBertCrossAttentionClassifier(nn.Module):
    def __init__(self, clip_model, clip_processor, bert_model, bert_tokenizer, num_classes: int):
        super().__init__()
        self.clip_model = clip_model
        self.clip_processor = clip_processor
        self.bert_model = bert_model
        self.bert_tokenizer = bert_tokenizer
        for m in [self.clip_model, self.bert_model]:
            for p in m.parameters():
                p.requires_grad = False
        self.clip_model.eval(); self.bert_model.eval()
        vd = clip_model.config.projection_dim
        td = bert_model.config.hidden_size
        self.fusion = CrossAttentionFusion(vd, td)
        self.classifier = build_classifier_head(CROSS_ATTN_DIM, num_classes)

    def encode(self, pixel_values, captions, zero_image: bool = False, zero_text: bool = False):
        dev = pixel_values.device
        with torch.no_grad():
            visual = self.clip_model.get_image_features(pixel_values=pixel_values).float()
            enc = self.bert_tokenizer(captions, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(dev)
            text_tokens = self.bert_model(**enc).last_hidden_state.float()
        if zero_image:
            visual = torch.zeros_like(visual)
        if zero_text:
            text_tokens = torch.zeros_like(text_tokens)
        pad_mask = enc["attention_mask"] == 0
        return visual, text_tokens, pad_mask, enc

    def forward(self, pixel_values, captions, return_attn: bool = False, zero_image: bool = False, zero_text: bool = False):
        visual, text_tokens, pad_mask, _ = self.encode(pixel_values, captions, zero_image, zero_text)
        if return_attn:
            fused, attn = self.fusion(visual, text_tokens, pad_mask, return_attn=True)
            return self.classifier(fused), attn
        fused = self.fusion(visual, text_tokens, pad_mask)
        return self.classifier(fused)


def collate_mm(batch, clip_processor):
    images = [Image.open(x["abs_path"]).convert("RGB") for x in batch]
    pixel_values = clip_processor(images=images, return_tensors="pt")["pixel_values"]
    return {"pixel_values": pixel_values, "captions": [x["caption"] for x in batch], "labels": torch.stack([x["labels"] for x in batch])}


def build_cross_attention_model() -> ClipBertCrossAttentionClassifier:
    clip_m, clip_p, bert_m, bert_t = get_clip_bert()
    return ClipBertCrossAttentionClassifier(clip_m, clip_p, bert_m, bert_t, num_classes)

print("Cross-attention model defined.")


### Model builders for evaluation

In [ ]:
# --- Image-only models ---
class FashionImageDataset(Dataset):
    def __init__(self, frame, style_to_idx, transform):
        self.frame = frame.reset_index(drop=True)
        self.style_to_idx = style_to_idx
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row["abs_path"]).convert("RGB")
        return {"pixel_values": self.transform(img), "labels": torch.tensor(self.style_to_idx[row["style"]], dtype=torch.long), "abs_path": row["abs_path"]}

class CLIPImageClassifier(nn.Module):
    def __init__(self, clip, num_classes):
        super().__init__(); self.clip = clip
        for p in self.clip.parameters(): p.requires_grad = False
        self.head = build_classifier_head(clip.config.projection_dim, num_classes)
    def forward(self, x):
        with torch.no_grad(): feats = self.clip.get_image_features(pixel_values=x).float()
        return self.head(feats)

class TimmImageClassifier(nn.Module):
    def __init__(self, backbone_name, num_classes):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0)
        self.head = build_classifier_head(self.backbone.num_features, num_classes)
    def forward(self, x): return self.head(self.backbone(x))

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_tf = lambda img: clip_processor(images=img, return_tensors="pt")["pixel_values"].squeeze(0)
vit_tf = timm.data.create_transform(**timm.data.resolve_data_config({}, model=VIT_MODEL_NAME), is_training=False)
conv_tf = timm.data.create_transform(**timm.data.resolve_data_config({}, model=CONVNEXT_MODEL_NAME), is_training=False)
swin_tf = timm.data.create_transform(**timm.data.resolve_data_config({}, model=SWIN_MODEL_NAME), is_training=False)

IMAGE_TRANSFORMS = {"clip": clip_tf, "vit": vit_tf, "convnext": conv_tf, "swin": swin_tf}


def build_image_only(model_key: str):
    if model_key == "clip":
        return CLIPImageClassifier(CLIPModel.from_pretrained(CLIP_MODEL_ID), num_classes)
    names = {"vit": VIT_MODEL_NAME, "convnext": CONVNEXT_MODEL_NAME, "swin": SWIN_MODEL_NAME}
    return TimmImageClassifier(names[model_key], num_classes)


# --- Multimodal baselines (minimal) ---
class ConcatMLPFusion(nn.Module):
    def __init__(self, vd, td): super().__init__(); self.mlp = nn.Sequential(nn.Linear(vd+td, 512), nn.ReLU(), nn.Dropout(DROPOUT))
    def forward(self, v, t): return self.mlp(torch.cat([v, t], dim=-1))

class GatedFusion(nn.Module):
    def __init__(self, vd, td):
        super().__init__()
        self.gate = nn.Sequential(nn.Linear(vd+td, 1), nn.Sigmoid())
        self.text_proj = nn.Linear(td, vd)
    def forward(self, v, t):
        g = self.gate(torch.cat([v, t], dim=-1))
        return g * v + (1 - g) * self.text_proj(t)

class ClipBertFusionClassifier(nn.Module):
    def __init__(self, clip_m, clip_p, bert_m, bert_t, fusion_type):
        super().__init__()
        self.clip_model, self.clip_processor, self.bert_model, self.bert_tokenizer = clip_m, clip_p, bert_m, bert_t
        for m in [self.clip_model, self.bert_model]:
            for p in m.parameters(): p.requires_grad = False
        vd, td = clip_m.config.projection_dim, bert_m.config.hidden_size
        self.fusion = ConcatMLPFusion(vd, td) if fusion_type == "concat_mlp" else GatedFusion(vd, td)
        head_in = 512 if fusion_type == "concat_mlp" else vd
        self.classifier = build_classifier_head(head_in, num_classes)
        self.fusion_type = fusion_type
    def forward(self, pixel_values, captions):
        dev = pixel_values.device
        with torch.no_grad():
            v = self.clip_model.get_image_features(pixel_values=pixel_values).float()
            enc = self.bert_tokenizer(captions, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(dev)
            t = self.bert_model(**enc).last_hidden_state[:, 0, :].float()
        return self.classifier(self.fusion(v, t))

class TextOnlyClassifier(nn.Module):
    def __init__(self, bert, tok):
        super().__init__(); self.bert, self.tokenizer = bert, tok
        for p in self.bert.parameters(): p.requires_grad = False
        self.head = build_classifier_head(768, num_classes)
    def forward(self, captions):
        dev = next(self.head.parameters()).device
        with torch.no_grad():
            enc = self.tokenizer(captions, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(dev)
            cls = self.bert(**enc).last_hidden_state[:, 0, :].float()
        return self.head(cls)

class BLIPFusionClassifier(nn.Module):
    def __init__(self, blip_model, num_classes):
        super().__init__()
        self.blip = blip_model
        for p in self.blip.parameters():
            p.requires_grad = False
        self.classifier = build_classifier_head(1024, num_classes)

    def forward(self, pixel_values, input_ids, attention_mask):
        with torch.no_grad():
            img = self.blip.get_image_features(pixel_values=pixel_values).float()
            txt = self.blip.get_text_features(input_ids=input_ids, attention_mask=attention_mask).float()
        return self.classifier(torch.cat([img, txt], dim=-1))


class ViltFusionClassifier(nn.Module):
    def __init__(self, vilt_model, num_classes):
        super().__init__()
        self.vilt = vilt_model
        for p in self.vilt.parameters():
            p.requires_grad = False
        self.classifier = build_classifier_head(vilt_model.config.hidden_size, num_classes)

    def forward(self, pixel_values, input_ids, attention_mask, token_type_ids=None):
        kwargs = dict(
            pixel_values=pixel_values,
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True,
        )
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        with torch.no_grad():
            out = self.vilt(**kwargs)
            pooled = out.pooler_output if out.pooler_output is not None else out.last_hidden_state[:, 0, :]
        return self.classifier(pooled.float())


print("Evaluation model builders ready.")


### 1 Confusion matrix — all baseline models (excluding proposed cross-attention)
For each baseline: aggregate confusion over all seeds, save heatmap and top confused pairs, then build a **whole summary** across models.


In [ ]:
def collate_image_batch(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "abs_paths": [x["abs_path"] for x in batch],
    }


def make_image_loader(test_df, model_key, seed):
    tf = IMAGE_TRANSFORMS[model_key]
    ds = FashionImageDataset(test_df, style_to_idx, tf)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_image_batch)


def make_mm_loader(test_df, seed):
    clip_p = get_clip_bert()[1]
    return DataLoader(FashionMultiModalFrameDataset(test_df, style_to_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_mm(b, clip_p))


def make_text_loader(test_df):
    return DataLoader(FashionMultiModalFrameDataset(test_df, style_to_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: {"captions": [x["caption"] for x in b], "labels": torch.stack([x["labels"] for x in b])})


all_top_pairs = []
global_summary = []

for model_key, (category, _) in ALL_BASELINE_MODELS.items():
    print("\n" + "=" * 60)
    print(f"Confusion matrix: {model_key} ({category})")
    agg_cm = np.zeros((num_classes, num_classes), dtype=np.int64)

    for seed in SEEDS:
        _, _, test_df = split_by_seed(df_mm if category != "image_only" else df_images, seed)
        if category == "image_only":
            model = build_image_only(model_key).to(device)
            load_weights(model, checkpoint_path(category, model_key, seed))
            loader = make_image_loader(test_df, model_key, seed)
            yt, yp = predict_loader(model, loader, lambda m, b: m(b["pixel_values"]))
        elif category == "multimodal":
            if model_key in {"concat_mlp", "gated_fusion"}:
                clip_m, clip_p, bert_m, bert_t = get_clip_bert()
                model = ClipBertFusionClassifier(clip_m, clip_p, bert_m, bert_t, model_key).to(device)
                load_weights(model, checkpoint_path(category, model_key, seed))
                loader = make_mm_loader(test_df, seed)
                yt, yp = predict_loader(model, loader, lambda m, b: m(b["pixel_values"], b["captions"]))
            elif model_key == "blip":
                blip_m = BlipModel.from_pretrained(BLIP_MODEL_ID)
                blip_p = BlipProcessor.from_pretrained(BLIP_MODEL_ID)
                model = BLIPFusionClassifier(blip_m, num_classes).to(device)
                load_weights(model, checkpoint_path(category, model_key, seed))

                def collate_blip(batch):
                    images = [Image.open(x["abs_path"]).convert("RGB") for x in batch]
                    caps = [x["caption"] for x in batch]
                    enc = blip_p(
                        images=images,
                        text=caps,
                        padding=True,
                        truncation=True,
                        max_length=MAX_SEQ_LENGTH,
                        return_tensors="pt",
                    )
                    enc["labels"] = torch.stack([x["labels"] for x in batch])
                    return enc

                loader = DataLoader(
                    FashionMultiModalFrameDataset(test_df, style_to_idx),
                    batch_size=BATCH_SIZE,
                    shuffle=False,
                    collate_fn=collate_blip,
                )
                yt, yp = predict_loader(model, loader, lambda m, b: m(b["pixel_values"], b["input_ids"], b["attention_mask"]))
            elif model_key == "vilbert":
                vilt_m = ViltModel.from_pretrained(VILT_MODEL_ID)
                vilt_p = ViltProcessor.from_pretrained(VILT_MODEL_ID)
                model = ViltFusionClassifier(vilt_m, num_classes).to(device)
                load_weights(model, checkpoint_path(category, model_key, seed))

                def collate_vilt(batch):
                    images = [Image.open(x["abs_path"]).convert("RGB") for x in batch]
                    caps = [x["caption"] for x in batch]
                    enc = vilt_p(
                        images=images,
                        text=caps,
                        padding=True,
                        truncation=True,
                        max_length=MAX_SEQ_LENGTH_VILT,
                        return_tensors="pt",
                    )
                    enc["labels"] = torch.stack([x["labels"] for x in batch])
                    return enc

                loader = DataLoader(
                    FashionMultiModalFrameDataset(test_df, style_to_idx),
                    batch_size=BATCH_SIZE,
                    shuffle=False,
                    collate_fn=collate_vilt,
                )
                yt, yp = predict_loader(
                    model,
                    loader,
                    lambda m, b: m(b["pixel_values"], b["input_ids"], b["attention_mask"], b.get("token_type_ids")),
                )
            else:
                continue
        else:
            model = TextOnlyClassifier(AutoModel.from_pretrained(BERT_MODEL_ID), AutoTokenizer.from_pretrained(BERT_MODEL_ID)).to(device)
            load_weights(model, checkpoint_path(category, model_key, seed))
            loader = make_text_loader(test_df)
            yt, yp = predict_loader(model, loader, lambda m, b: m(b["captions"]))

        cm = confusion_matrix(yt, yp, labels=list(range(num_classes)))
        agg_cm += cm
        acc = accuracy_score(yt, yp)
        print(f"  seed {seed}: acc={acc:.4f}")

    if agg_cm.sum() == 0:
        continue

    out_dir = DIR_CONFUSION / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "aggregated_confusion_matrix.npy", agg_cm)
    plot_confusion_heatmap(agg_cm, f"Aggregated confusion — {model_key}", out_dir / "confusion_matrix_heatmap.png")

    top_df = top_confused_pairs(agg_cm)
    top_df.insert(0, "model", model_key)
    top_df.to_csv(out_dir / "top_confused_pairs.csv", index=False)
    print(top_df)
    all_top_pairs.append(top_df)

    global_summary.append({
        "model": model_key,
        "category": category,
        "total_errors": int(agg_cm.sum() - np.trace(agg_cm)),
        "accuracy": float(np.trace(agg_cm) / max(agg_cm.sum(), 1)),
        "macro_f1_off_diag_proxy": float(1 - (agg_cm.sum() - np.trace(agg_cm)) / max(agg_cm.sum(), 1)),
    })

if all_top_pairs:
    pd.concat(all_top_pairs, ignore_index=True).to_csv(DIR_CONFUSION / "all_models_top_confused_pairs.csv", index=False)
df_global = pd.DataFrame(global_summary)
df_global.to_csv(DIR_CONFUSION / "whole_summary_all_baselines.csv", index=False)
print("\n=== Whole summary (all baseline models) ===")
print(df_global)

### B.2 Class-pair confusion (cross-attention + image-only)
Top confused class pairs and **symmetric** confusion rates.


In [ ]:
pair_summary_all = []

for model_key in CLASS_PAIR_MODELS:
    print("\n" + "=" * 60)
    print(f"Class-pair analysis: {model_key}")
    agg_cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    category = "proposed" if model_key == PROPOSED_MODEL_KEY else "image_only"

    for seed in SEEDS:
        _, _, test_df = split_by_seed(df_mm if model_key == PROPOSED_MODEL_KEY else df_images, seed)
        if model_key == PROPOSED_MODEL_KEY:
            model = build_cross_attention_model().to(device)
            load_weights(model, checkpoint_path("proposed", model_key, seed))
            loader = DataLoader(FashionMultiModalFrameDataset(test_df, style_to_idx), batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_mm(b, get_clip_bert()[1]))
            yt, yp = predict_loader(model, loader, lambda m, b: m(b["pixel_values"], b["captions"]))
        else:
            model = build_image_only(model_key).to(device)
            load_weights(model, checkpoint_path("image_only", model_key, seed))
            loader = make_image_loader(test_df, model_key, seed)
            yt, yp = predict_loader(model, loader, lambda m, b: m(b["pixel_values"]))
        agg_cm += confusion_matrix(yt, yp, labels=list(range(num_classes)))

    out_dir = DIR_CLASS_PAIR / model_key
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "aggregated_confusion_matrix.npy", agg_cm)

    top_df = top_confused_pairs(agg_cm)
    sym_df = symmetric_confusion_rates(agg_cm)
    top_df.to_csv(out_dir / "top_confused_pairs.csv", index=False)
    sym_df.to_csv(out_dir / "symmetric_confusion.csv", index=False)
    print("Top confused pairs:")
    print(top_df.head(10))
    print("Symmetric confusion (top 10):")
    print(sym_df.head(10))

    plot_confusion_heatmap(agg_cm, f"Symmetric class-pair view — {model_key}", out_dir / "confusion_heatmap.png")
    top_pair = ""
    top_count = 0
    if len(top_df):
        top_pair = f"{top_df.iloc[0]['true_class']}->{top_df.iloc[0]['pred_class']}"
        top_count = int(top_df.iloc[0]["count"])
    pair_summary_all.append({"model": model_key, "top_pair": top_pair, "top_count": top_count})

pd.DataFrame(pair_summary_all).to_csv(DIR_CLASS_PAIR / "class_pair_summary.csv", index=False)
print("\nSaved class-pair analysis under", DIR_CLASS_PAIR)


### B.3 Grad-CAM on CLIP vision encoder (image-only + cross-attention)
Grad-CAM targets the last CLIP ViT encoder block for **successful** and **misclassified** test examples. Runs for **every seed** in `SEEDS`; outputs: `gradcam/<model>/seed_<seed>/`.


In [ ]:
def validate_gradcam_prerequisites() -> None:
    """Verify dataset, checkpoint, and output paths before Grad-CAM runs."""
    required_files = [
        COMPLETE_CSV,
        CAPTION_CSV,
        DATA_DIR / "seeds_list.txt",
    ]
    missing_files = [p for p in required_files if not p.is_file()]
    missing_ckpts = []
    for seed in SEEDS:
        for model_key in GRADCAM_MODELS:
            category = "proposed" if model_key == PROPOSED_MODEL_KEY else "image_only"
            ckpt = checkpoint_path(category, model_key, seed)
            if not ckpt.is_file():
                missing_ckpts.append(str(ckpt))
    for d in [DIR_GRADCAM, OUTPUT_ROOT]:
        d.mkdir(parents=True, exist_ok=True)
    if missing_files:
        raise FileNotFoundError("Missing required files:\n  " + "\n  ".join(map(str, missing_files)))
    if missing_ckpts:
        raise FileNotFoundError(
            "Missing Grad-CAM checkpoints (re-run training or adjust SEEDS):\n  "
            + "\n  ".join(missing_ckpts)
        )
    print("Grad-CAM prerequisites OK.")
    print(f"  Models: {GRADCAM_MODELS}")
    print(f"  Seeds: {SEEDS}")
    print(f"  Output: {DIR_GRADCAM.resolve()}")


class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None

        def _first_tensor(x):
            return x[0] if isinstance(x, tuple) else x

        def fwd_hook(_, __, output):
            out = _first_tensor(output)
            if torch.is_grad_enabled():
                out.retain_grad()
            self.activations = out

        def bwd_hook(_, grad_input, grad_output):
            g = grad_output[0] if isinstance(grad_output, (tuple, list)) else grad_output
            self.gradients = g.detach() if g is not None else None

        target_layer.register_forward_hook(fwd_hook)
        target_layer.register_full_backward_hook(bwd_hook)

    def _logits_for_cam(self, pixel_values, captions=None):
        if hasattr(self.model, "clip") and captions is None:
            clip = self.model.clip
            vision_out = clip.vision_model(pixel_values=pixel_values)
            pooled = vision_out[1]
            feats = clip.visual_projection(pooled).float()
            return self.model.head(feats)
        if hasattr(self.model, "clip_model") and captions is not None:
            clip = self.model.clip_model
            bert = self.model.bert_model
            tok = self.model.bert_tokenizer
            dev = pixel_values.device
            vision_out = clip.vision_model(pixel_values=pixel_values)
            pooled = vision_out[1]
            visual = clip.visual_projection(pooled).float()
            enc = tok(captions, return_tensors="pt", padding=True, truncation=True, max_length=MAX_SEQ_LENGTH).to(dev)
            with torch.no_grad():
                text_tokens = bert(**enc).last_hidden_state.float()
            pad_mask = enc["attention_mask"] == 0
            fused = self.model.fusion(visual, text_tokens, pad_mask)
            return self.model.classifier(fused)
        if hasattr(self.model, "clip"):
            return self.model(pixel_values)
        if captions is not None:
            return self.model(pixel_values, captions)
        raise ValueError("Captions required for cross-attention Grad-CAM")

    def generate(self, pixel_values, target_class: int, captions=None) -> np.ndarray:
        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None
        out_size = pixel_values.shape[-2:]
        pv = pixel_values.detach()
        if not pv.requires_grad:
            pv = pv.requires_grad_(True)
        with torch.enable_grad():
            logits = self._logits_for_cam(pv, captions)
            score = logits[0, target_class]
            score.backward()
        acts = self.activations
        grads = self.gradients
        if grads is None and acts is not None and acts.grad is not None:
            grads = acts.grad
        if acts is None or grads is None:
            raise RuntimeError(
                "Grad-CAM could not capture gradients on the target layer. "
                "Check that CLIP vision forward runs with grad-enabled inputs."
            )
        if acts.dim() == 4:
            weights = grads.mean(dim=(2, 3), keepdim=True)
            cam = (weights * acts).sum(dim=1, keepdim=True)
            cam = F.relu(cam)
        elif acts.dim() == 3:
            weights = grads.mean(dim=-1, keepdim=True)
            cam = (weights * acts).sum(dim=-1)
            cam = F.relu(cam[:, 1:])
            side = int(cam.shape[1] ** 0.5)
            cam = cam.view(1, 1, side, side)
        else:
            raise ValueError(f"Unsupported activation shape: {tuple(acts.shape)}")
        cam = F.interpolate(cam, size=out_size, mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


def get_clip_vision_module(model):
    if hasattr(model, "clip_model"):
        return model.clip_model.vision_model.encoder.layers[-1]
    return model.clip.vision_model.encoder.layers[-1]


def overlay_cam(image: Image.Image, cam: np.ndarray, out_path: Path, title: str):
    img = np.array(image.resize((224, 224))) / 255.0
    heat = plt.cm.jet(cam)[:, :, :3]
    blended = 0.5 * img + 0.5 * heat
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img); axes[0].set_title("Image"); axes[0].axis("off")
    axes[1].imshow(cam, cmap="jet"); axes[1].set_title("Grad-CAM"); axes[1].axis("off")
    axes[2].imshow(blended); axes[2].set_title(title); axes[2].axis("off")
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.show(); plt.close(fig)
    print(f"Saved Grad-CAM: {out_path}")


validate_gradcam_prerequisites()
all_gradcam_rows = []

for seed in SEEDS:
    print("\n" + "#" * 60)
    print(f"Grad-CAM | seed {seed}")
    _, _, test_df_grad = split_by_seed(df_mm, seed)

    for model_key in GRADCAM_MODELS:
        print("\n" + "=" * 60)
        print(f"Grad-CAM: {model_key} | seed {seed}")
        out_dir = DIR_GRADCAM / model_key / f"seed_{seed}"
        out_dir.mkdir(parents=True, exist_ok=True)

        if model_key == PROPOSED_MODEL_KEY:
            model = build_cross_attention_model().to(device)
            load_weights(model, checkpoint_path("proposed", model_key, seed))

            def collate_mm_grad(batch):
                out = collate_mm(batch, get_clip_bert()[1])
                out["abs_paths"] = [x["abs_path"] for x in batch]
                return out

            loader = DataLoader(
                FashionMultiModalFrameDataset(test_df_grad, style_to_idx),
                batch_size=1,
                shuffle=False,
                collate_fn=collate_mm_grad,
            )
        else:
            model = build_image_only(model_key).to(device)
            load_weights(model, checkpoint_path("image_only", model_key, seed))
            loader = make_image_loader(test_df_grad, model_key, seed)

        target_layer = get_clip_vision_module(model)
        gradcam = GradCAM(model, target_layer)

        success_cases, failure_cases = [], []
        for batch in loader:
            pv = batch["pixel_values"].to(device)
            if pv.dim() == 3:
                pv = pv.unsqueeze(0)
            labels = batch["labels"]
            if labels.dim() == 0:
                labels = labels.unsqueeze(0)
            with torch.no_grad():
                if model_key == PROPOSED_MODEL_KEY:
                    logits = model(pv, batch["captions"])
                else:
                    logits = model(pv)
            preds = logits.argmax(dim=1)
            for i in range(pv.size(0)):
                label = int(labels[i].item())
                pred = int(preds[i].item())
                abs_path = None
                if "abs_paths" in batch:
                    abs_path = batch["abs_paths"][i]
                elif "abs_path" in batch:
                    ap = batch["abs_path"]
                    abs_path = ap[i] if isinstance(ap, list) else ap
                caps = batch.get("captions")
                if caps is not None and isinstance(caps, list):
                    caps = [caps[i]]
                rec = {
                    "true": idx_to_style[label],
                    "pred": idx_to_style[pred],
                    "pv": pv[i : i + 1],
                    "label": label,
                    "pred_idx": pred,
                    "abs_path": abs_path,
                    "captions": caps,
                }
                if pred == label:
                    success_cases.append(rec)
                else:
                    failure_cases.append(rec)
                if len(success_cases) >= GRADCAM_NUM_CASES and len(failure_cases) >= GRADCAM_NUM_CASES:
                    break
            if len(success_cases) >= GRADCAM_NUM_CASES and len(failure_cases) >= GRADCAM_NUM_CASES:
                break

        rows = []
        for case_type, cases in [("success", success_cases[:GRADCAM_NUM_CASES]), ("failure", failure_cases[:GRADCAM_NUM_CASES])]:
            for i, c in enumerate(cases):
                model.train()
                caps = c.get("captions") if model_key == PROPOSED_MODEL_KEY else None
                cam = gradcam.generate(c["pv"], c["pred_idx"], captions=caps)
                model.eval()
                img_path = c.get("abs_path")
                if not img_path or not Path(img_path).is_file():
                    raise FileNotFoundError(f"Image not found for Grad-CAM overlay: {img_path}")
                img = Image.open(img_path).convert("RGB")
                title = f"seed {seed} | {case_type}: true={c['true']} pred={c['pred']}"
                png = out_dir / f"{case_type}_{i:02d}.png"
                overlay_cam(img, cam, png, title)
                rows.append({
                    "seed": seed,
                    "model": model_key,
                    "case_type": case_type,
                    "true": c["true"],
                    "pred": c["pred"],
                    "file": str(png.relative_to(DIR_GRADCAM)),
                })

        pd.DataFrame(rows).to_csv(out_dir / "gradcam_cases.csv", index=False)
        print(pd.DataFrame(rows))
        all_gradcam_rows.extend(rows)

pd.DataFrame(all_gradcam_rows).to_csv(DIR_GRADCAM / "all_seeds_gradcam_cases.csv", index=False)
print("\nAll explainability and error-analysis outputs written under:", OUTPUT_ROOT.resolve())